In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

!wget -q -O X_dev.csv "https://raw.githubusercontent.com/mclarenh/Machine-Learning-Final-project/refs/heads/main/data/processed/X_dev.csv"
!wget -q -O y_dev.csv "https://raw.githubusercontent.com/mclarenh/Machine-Learning-Final-project/refs/heads/main/data/processed/y_dev.csv"
!wget -q -O shared.py "https://raw.githubusercontent.com/mclarenh/Machine-Learning-Final-project/refs/heads/main/shared.py"

from shared import accuracy

X_dev = pd.read_csv("X_dev.csv").to_numpy(dtype=float)
y_dev = pd.read_csv("y_dev.csv").iloc[:, 0].to_numpy(dtype=int)

print("Development rows:", len(y_dev))
print("Features:", X_dev.shape[1])

SEED = 42

folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_dev, y_dev))

def majority(labels):
    """The most common class label; a tie goes to the smaller label."""
    return np.bincount(labels).argmax()

def entropy(y):
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def best_split(X, y):
    n = len(y)
    best_j, best_s, best_E = None, None, np.inf
    for j in range(X.shape[1]):                     # every feature
        values = np.unique(X[:, j])                 # sorted distinct values
        for s in (values[:-1] + values[1:]) / 2:    # every midpoint
            y_left = y[X[:, j] < s]
            y_right = y[X[:, j] >= s]
            E = (len(y_left)/n) * entropy(y_left) + (len(y_right)/n) * entropy(y_right)   # TODO: weighted entropy of this split (one line)
            if E < best_E:
                best_j, best_s, best_E = j, s, E
    return best_j, best_s, best_E

def build_tree(X, y, max_depth=None, depth=0):
    """Grow a tree greedily. max_depth=None means no depth limit."""
    node = {'value': majority(y)}                   # every node stores its majority class
    if entropy(y) == 0 or depth == max_depth:       # pure node or depth limit: make a leaf
        return node
    j, s, E = best_split(X, y)
    if j is None:                                   # no split is possible: make a leaf
        return node
    go_right = X[:, j] >= s
    node['feature'] = j
    node['threshold'] = s
    node['left'] = build_tree(X[~go_right], y[~go_right], max_depth, depth + 1)
    node['right'] = build_tree(X[go_right], y[go_right], max_depth, depth + 1)
    return node

def predict_tree(tree, X):
    predictions = []
    for x in X:
        node = tree
        while 'feature' in node:                    # walk down until we reach a leaf
            if x[node['feature']] < node['threshold']:
                node = node['left']
            else:
                node = node['right']
        predictions.append(node['value'])
    return np.array(predictions)

def cv_tree(X, y, max_depth):
    """5-fold CV accuracy of a tree. Trees need no scaling."""
    scores = []
    for train_rows, val_rows in folds:
        tree = build_tree(X[train_rows], y[train_rows], max_depth)
        y_pred = predict_tree(tree, X[val_rows])
        scores.append(accuracy(y[val_rows], y_pred))
    return np.mean(scores), np.std(scores)

depths = [1, 3, 5]

tree_train, tree_cv, tree_sd = [], [], []
for d in depths:
    # Training accuracy: train on all development rows and predict those same rows.
    tree = build_tree(X_dev, y_dev, max_depth=d)
    tree_train.append(accuracy(y_dev, predict_tree(tree, X_dev)))       # TODO: training accuracy of this tree on X_dev
    # CV accuracy: each fold trains on 4/5 of the rows and predicts the held-out 1/5.
    cv_mean, cv_sd = cv_tree(X_dev, y_dev, d)   # TODO: 5-fold CV accuracy with max_depth = d (use cv_tree)
    tree_cv.append(cv_mean)
    tree_sd.append(cv_sd)

# Choose the depth with the highest CV accuracy.
# depths is in increasing order, so ">" keeps the shallower depth when accuracies tie.
best_depth, best_acc = depths[0], tree_cv[0]
for d, acc in zip(depths, tree_cv):
    if acc > best_acc:
        best_depth, best_acc = d, acc

print('Selected max_depth:', best_depth)
pd.DataFrame({'max_depth': [str(d) for d in depths], 'train_acc': tree_train,
              'CV_acc': tree_cv, 'CV_sd': tree_sd}).round(4)


Development rows: 734
Features: 21
Selected max_depth: 3


,max_depth,train_acc,CV_acc,CV_sd
0,1,0.8270,0.8270,0.0387
1,3,0.8542,0.8324,0.0344
2,5,0.8815,0.8229,0.0304
